## Fox News Scraper and Crawler 

This notebook contains the code for the Fox news scrapper and crawler. 

## Imports 

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.chrome.options import Options
import datetime
import random
import time
import json
import csv

## Crawler 

For the crawler we search terms "Israel", "Palestine", "Gaza", "Hamas",and "Conflict", we select the start date and end date both to be the current date that we are interested in. We select only articles as the media we are interested in. 

The Fox news website allows users to specify start date range and end date range. However the page limits the amount of articles we can view to 50-100, no specific rule. Found out by exploring the website. So the best way to scrape the maximum amount of articles possible was by using the same date for start date and end date range and extracting all the articles possible for that given day. We repeat this procedure for all the dates in our date range which starts from 10/7/2023 and ends at 12/31/2023. 

In our scraping practices we also skip articles tagged "Live-News" articles which include live news coverage summaries, these are similar to live-blogs observed by the cnn crawler. 

### Helper functions used by the crawler

In [ ]:
def extract_str_date_time(date_time_obj):
    """
    Extract month, date, and year and convert it into a format that can be 
    used as search values to put in our min and max date ranges. 
    We return the month, day, year information from a date time object 
    as a list. 

    INPUT: 
        date_time_obj (datetime obj): The current date we want to extract articles from 
    
    Return:
        return_list (list): [month (str), date (str), year (str)]
    """
    month = str(date_time_obj.month)
    day = str(date_time_obj.day) 
    year = str(date_time_obj.year)
    return_list  = []

    for date_comp in [month, day, year]: 
        if len(date_comp) == 1:
            date_comp = "0"+date_comp
            return_list.append(date_comp)
        else: 
            return_list.append(date_comp)

    return return_list

# Define a function to select date components for "min" and "max" date sections
def select_date_component(date_section, component, value, wait):
    """ 
    Selects the date section (min date, max date), the corresponding 
    component (month, day, year) and inputs the desired value for those 
    components using the options selector

    INPUT: 
        date_section (str): (min_date, max_date) which section to look at 
        component (str): (month, day, year) 
        value (str): the value inserted for that given component
        wait (chromedriver obj): used to search and click

    """
    button_selector = f'div.filter.date-range > div.date.{date_section} > div.sub.{component} > button.select'
    option_selector = f'div.filter.date-range > div.date.{date_section} > div.sub.{component} > ul.option > li[id="{value}"]'

    # Click to reveal options
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, button_selector))).click()
    # Select specific value
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, option_selector))).click()



def dict_keys_datetime_to_string(original_dict, date_format='%Y-%m-%d'):
    """
    Converts the keys of a dictionary from datetime objects to strings.
    
    Args:
    - original_dict: Dictionary with datetime object keys.
    - date_format: String format to convert datetime objects to strings.
    
    Returns:
    - A new dictionary with keys as strings.
    """
    return {key.strftime(date_format): value for key, value in original_dict.items()}




## Iterative crawling over dates

Crawl over all the dates between 10/07/2023 - 12/31/2023. Run into issues crawling the website in headless mode (no chrome window pop up ) so the current way shows a chrome window, might have to do with the dynamics of the website. 

In [ ]:
# Setup WebDriver with Chrome
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

temp_article_list = [] #to store all the articles for a given date
fox_articles_dict = {} #the keys are dates, and values are lists of links [] 

start_date = datetime.date(2023, 10, 7) #start date 
end_date = datetime.date(2023, 12, 31) #end date 

n_days  = (end_date - start_date).days + 1 #number of days between our dates 

#Iterating over each datetime object
for date in (start_date + datetime.timedelta(n) for n in range(n_days)): 
    
    sleep_time = random.randint(2,5)
    time.sleep(sleep_time)
    month, day, year = extract_str_date_time(date)

    # Navigate to the webpage
    article = "https://www.foxnews.com/search-results/search?q=Israel%20Hamas%20Gaza%20Palestine%20Conflict%20"
    driver.get(article)

    # Initialize WebDriverWait
    wait = WebDriverWait(driver, 3)
    # Click "Select Content Type" button and select "Article"
    content_type_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.filter.content > button.select')))
    content_type_button.click()
    # Select "Article" checkbox
    article_checkbox = wait.until(EC.element_to_be_clickable((By.XPATH, '//input[@title="Article"]')))
    article_checkbox.click()

    # Scroll to the bottom of the page to ensure all elements are loaded before selecting dates
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # Give time for any lazy-loaded elements to appear

    # Select date components for "date min"
    select_date_component("min", "month", month, wait)
    select_date_component("min", "day", day, wait)
    select_date_component("min", "year", year, wait)

    # Select date components for "date max"
    select_date_component("max", "month", month, wait)
    select_date_component("max", "day", day, wait)
    select_date_component("max", "year", year, wait)

    # Click the "Search" div using the CSS selector
    search_div = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#wrapper > div.search-container > div.search-directive > div > div.main-search > div.search-form > div")))
    search_div.click()

    #After search is complete, we scroll to the bottom of the page and repeatedly 
    #click the "Load More" till we all the articles for a given date are loaded 

    # Code for clicking "Load More" and extracting articles continues here...
    # Repeatedly click the "Load More" button until it's no longer found
    while True:
        try:
            # Scroll to the bottom of the page
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            # Wait for the "Load More" button to become clickable
            load_more_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#wrapper > div.search-container > div.search-results-content > div > div.collection.collection-search.active > div.button.load-more"))) 
            # Scroll into view of the Load More button explicitly
            driver.execute_script("arguments[0].scrollIntoView(true);", load_more_button)
            
            # Click the "Load More" button
            load_more_button.click()
            
            # Wait a moment for the page to load more content
            time.sleep(2)  # Adjust sleep time as necessary based on your page's load time
            
        except TimeoutException:
            # If the "Load More" button is not found, exit the loop
            print("No more 'Load More' button to click.")
            break
    
    
    #Once all the articles for a given date are loaded we extract them         
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.article")))

    # Locate all articles
    articles = driver.find_elements(By.CSS_SELECTOR, "article.article")
    
    # Iterate through each article
    for article in articles:
        # Try to find the eyebrow element within the article
        eyebrows = article.find_elements(By.CSS_SELECTOR, "span.eyebrow > a")
        # Check if eyebrow element exists and its text is not "live news"
        for eyebrow in eyebrows:
            if eyebrow.text.lower() != "live-news": #we ignore live-news articles
                href_link = eyebrow.get_attribute('href')
                #article_links.append(str(href_link))
                temp_article_list.append(str(href_link))
                
            else:
                print("Eyebrow text is 'live news', link skipped.")

    fox_articles_dict[date] = temp_article_list
    temp_article_list = [] 

driver.quit()

### Uploading Article links into a JSON file

Good practice in case the code bombs

In [ ]:
updated_article_dict = dict_keys_datetime_to_string(fox_articles_dict)
filename = "fox_news_links.json"

with open(filename, 'w') as file:
    json.dump(updated_article_dict, file, indent=4)

print(f"Dictionary saved to {filename}")

## Scraping Fox News Articles

### Loading Article links from a Json (optional)
Loading the json file. Optional, if the code above ran, no need to do this 

In [ ]:
filename = 'fox_news_links.json'

# Open the file and load its content into a Python dictionary
with open(filename, 'r') as file:
    fox_articles_dict = json.load(file)

### Scraper code

In [ ]:
# Function to scrape a single article information
def scrape_fox_articles(article_url, article_date, article_index, driver):
    """
    Scrapes the article body and title and returns a dictionary with article info

    INPUT: 
        article_url (str): Fox news articles to be scraped 
        article_date (str): Date of the article, mostly for datalogging purposes
        article_index (int): used in data logging to keep track the number of 
                            articles successfully scraped 
        driver (chromedriver obj): used to request the html of the article and 
                            dynamcally scrape its contents 
    
    RETURN: 
        article_info_dict (dict): dictionary with all necessary article info
    """
    
    print(f"Scraping article for date: {article_date}, URL: {article_url}, article_index = {article_index}")
    try:
        driver.get(article_url)
        wait = WebDriverWait(driver, 1)  # Adjusted wait time for better reliability

        #
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        article_title = driver.title.split("|")[0]
        print(f"Article title: {article_title}")

        xpath_selector = ".//div[contains(@class, 'article-body')]/p[not(.//a[@href])]"
        paragraphs = wait.until(EC.presence_of_all_elements_located((By.XPATH, xpath_selector)))
        all_text = [paragraph.text for paragraph in paragraphs if paragraph.text.strip()]
        if all_text:  # Check if list is not empty
            all_text.pop()  # Remove the last element cautiously
        all_text_str = " ".join(all_text)
        print("Successfully retrieved article content.")
        
        article_info_dict = {
            "article_url": article_url,
            "article_date": article_date,
            "article_title": article_title,
            "all_text": all_text_str
        }
        return article_info_dict
    except Exception as e:
        print(f"An error occurred for URL: {article_url} - {str(e)}, date = {article_date}, index = {article_index}")
        return None



### Iterative Crawling over the fox_news_dict

In [ ]:
# Setup Chrome options for headless mode
options = Options()
options.add_argument("--headless")  # Runs Chrome in headless mode.
options.add_argument('--no-sandbox')  # # Bypass OS security model
options.add_argument('--disable-gpu')  # applicable to windows os only
options.add_argument('start-maximized')  
options.add_argument('disable-infobars')
options.add_argument('--disable-extensions')


error_urls = [] #keeps track of the urls not scraped
count = 0 #tracks the number of articles scaped 
restart_driver_count = 10 #restart the driver every 10 urls scraped
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options = options)

#uncomment both error_point and error_count if code breaks 
#set the error point to the last count index 
#to the error 
#error_point = 130 
#error_count = 0 


# Initialize the CSV file for writing
csv_filename = 'fox_scraped_articles5.csv'
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    
    # The fieldnames will be updated inside the loop
    writer = csv.DictWriter(csvfile, fieldnames=[], quoting=csv.QUOTE_ALL)  # Ensure all fields are quoted
    writer.writeheader()  # Placeholder header, will be updated

    for date, article_urls in fox_articles_dict.items():
        for index, article_url in enumerate(article_urls):
            

            #uncomment the code below if you run into issues
            # error_count+=1 
            # if error_count <= error_point: 
            #     continue

            #We restart the driver every 10 iterations
            if count % restart_driver_count == 0 and count != 0: 
                driver.quit()
                driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

            article_info = scrape_fox_articles(article_url, date, index, driver)
            
            if article_info:
                # Update the writer with the correct fieldnames based on the first article
                if count == 0:
                    writer.fieldnames = article_info.keys()
                    writer.writeheader()  # Write the correct header based on the first article info
                writer.writerow(article_info)
            else:
                error_urls.append(article_url)      
            count += 1

driver.quit()
